# Phase 16 — Component Ablation Study (FiLM x Contraction-Delta)

**Why this matters for the paper:** the final model (notebook 15) combines two architectural additions on top of the plain [f_ED, f_ES] concatenation -- the explicit contraction-delta feature (f_ED - f_ES) and the Quality-FiLM modulation. So far we have no evidence isolating each component's individual contribution; a reviewer will ask exactly this. This notebook trains all 4 combinations (2x2) on a short schedule and compares them **on VAL only** -- the test set is not touched again, preserving the integrity of the already-reported final test numbers.

**The 4 configurations:**
1. Baseline: [f_ED, f_ES] only, no delta, no FiLM
2. + Contraction-delta: [f_ED, f_ES, f_ED-f_ES], no FiLM
3. + FiLM only: [f_ED, f_ES], with FiLM
4. Full model (as in notebook 15): [f_ED, f_ES, f_ED-f_ES], with FiLM

Reuses the ED/ES frame cache and SimCLR-pretrained backbone from notebooks 14/07 -- no new data or pretraining needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, mean_absolute_error

PROJECT_ROOT = '/content/drive/MyDrive/echonet_project'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data/raw', 'EchoNet-Dynamic', 'EchoNet-Dynamic')
FULL_CSV = os.path.join(DATA_DIR, 'FileList.csv')
EDES_CACHE_DIR = os.path.join(PROJECT_ROOT, 'data/edes', 'frames')
quality_csv_path = os.path.join(PROJECT_ROOT, 'data/full', 'quality_scores.csv')
PRETRAINED_BACKBONE_PATH = os.path.join(PROJECT_ROOT, 'checkpoints', 'simclr_pretrain_v1', 'backbone_only.pt')

assert os.path.isdir(EDES_CACHE_DIR) and len(os.listdir(EDES_CACHE_DIR)) > 9000, 'Run 14_ed_es_extraction.ipynb first.'

df = pd.read_csv(FULL_CSV)
quality_df = pd.read_csv(quality_csv_path)
ef_col = 'EF' if 'EF' in df.columns else [c for c in df.columns if 'ef' in c.lower()][0]
filename_col = 'FileName' if 'FileName' in df.columns else [c for c in df.columns if 'file' in c.lower() and 'name' in c.lower()][0]
HFrEF_THRESHOLD = 40

for col in ['sharpness', 'contrast']:
    lo, hi = quality_df[col].quantile(0.02), quality_df[col].quantile(0.98)
    quality_df[col + '_norm'] = ((quality_df[col] - lo) / max(hi - lo, 1e-6)).clip(0, 1)
quality_df['quality_score'] = quality_df[['sharpness_norm', 'contrast_norm']].mean(axis=1)
df = df.merge(quality_df[[filename_col, 'quality_score']], on=filename_col, how='left')

valid_mask = df[filename_col].apply(lambda f: os.path.exists(
    os.path.join(EDES_CACHE_DIR, (str(f) if str(f).endswith('.avi') else str(f)+'.avi').replace('.avi', '.npy'))))
clean_df = df[valid_mask & df['quality_score'].notna()].reset_index(drop=True)
train_df = clean_df[clean_df['Split'].str.upper() == 'TRAIN'].reset_index(drop=True)
val_df = clean_df[clean_df['Split'].str.upper() == 'VAL'].reset_index(drop=True)
print(f'Train: {len(train_df)}  Val: {len(val_df)} (TEST is not used in this notebook)')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: enable GPU -- 4 training runs will be slow on CPU.')

In [ ]:
class EchoEDESDataset(Dataset):
    def __init__(self, df, frame_dir, filename_col, ef_col, train=True):
        self.df = df.reset_index(drop=True)
        self.frame_dir = frame_dir
        self.filename_col = filename_col
        self.ef_col = ef_col
        self.train = train
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = str(row[self.filename_col])
        if not fname.endswith('.avi'):
            fname = fname + '.avi'
        pair = np.load(os.path.join(self.frame_dir, fname.replace('.avi', '.npy')))
        pair = torch.from_numpy(pair).permute(0, 3, 1, 2).float() / 255.0
        pair = F.interpolate(pair, size=(112, 112), mode='bilinear', align_corners=False)
        if self.train and torch.rand(1).item() < 0.5:
            pair = torch.flip(pair, dims=[3])
        pair = (pair - self.mean) / self.std
        ef = float(row[self.ef_col])
        label = 1.0 if ef < HFrEF_THRESHOLD else 0.0
        quality = float(row['quality_score']) if not pd.isna(row['quality_score']) else 0.5
        return pair, torch.tensor(ef, dtype=torch.float32), torch.tensor(label, dtype=torch.float32), torch.tensor(quality, dtype=torch.float32)

BATCH_SIZE = 24
train_ds = EchoEDESDataset(train_df, EDES_CACHE_DIR, filename_col, ef_col, train=True)
val_ds = EchoEDESDataset(val_df, EDES_CACHE_DIR, filename_col, ef_col, train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## Configurable model: contraction-delta and FiLM can each be toggled off

In [ ]:
class QualityFiLM(nn.Module):
    def __init__(self, feat_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, 2 * feat_dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)
        self.feat_dim = feat_dim

    def forward(self, quality_score):
        params = self.net(quality_score.unsqueeze(-1))
        gamma, beta = params[:, :self.feat_dim], params[:, self.feat_dim:]
        return 1.0 + gamma, beta

class EchoAblation(nn.Module):
    def __init__(self, use_delta, use_film, dropout_p=0.2, pretrained_backbone_path=None):
        super().__init__()
        backbone = models.resnet18(weights=None)
        backbone.fc = nn.Identity()
        if pretrained_backbone_path:
            backbone.load_state_dict(torch.load(pretrained_backbone_path, map_location='cpu'))
        self.backbone = backbone
        self.use_delta = use_delta
        self.use_film = use_film
        feat_dim = 512 * 3 if use_delta else 512 * 2
        if use_film:
            self.film = QualityFiLM(feat_dim=feat_dim)
        self.dropout = nn.Dropout(p=dropout_p)
        self.ef_mean_head = nn.Linear(feat_dim, 1)
        self.ef_q_lo_head = nn.Linear(feat_dim, 1)
        self.ef_q_hi_head = nn.Linear(feat_dim, 1)
        self.cls_head = nn.Linear(feat_dim, 1)

    def forward(self, pair, quality_score):
        ed = pair[:, 0]
        es = pair[:, 1]
        f_ed = self.backbone(ed)
        f_es = self.backbone(es)
        if self.use_delta:
            feats = torch.cat([f_ed, f_es, f_ed - f_es], dim=1)
        else:
            feats = torch.cat([f_ed, f_es], dim=1)
        if self.use_film:
            gamma, beta = self.film(quality_score)
            feats = gamma * feats + beta
        feats = self.dropout(feats)
        ef_mean = self.ef_mean_head(feats).squeeze(-1)
        q_lo = self.ef_q_lo_head(feats).squeeze(-1)
        q_hi = self.ef_q_hi_head(feats).squeeze(-1)
        cls_logit = self.cls_head(feats).squeeze(-1)
        return ef_mean, q_lo, q_hi, cls_logit

def pinball_loss(y_true, y_pred, tau):
    diff = y_true - y_pred
    return torch.mean(torch.max(tau * diff, (tau - 1) * diff))

TAU_LO, TAU_HI = 0.05, 0.95

## Reusable short training + eval function for one ablation configuration

In [ ]:
def evaluate(model, loader):
    model.eval()
    ef_true_all, ef_pred_all, cls_true_all, cls_prob_all = [], [], [], []
    with torch.no_grad():
        for pair, ef_true, cls_true, quality in loader:
            pair, quality = pair.to(device), quality.to(device)
            ef_mean, q_lo, q_hi, cls_logit = model(pair, quality)
            ef_true_all.extend(ef_true.numpy()); ef_pred_all.extend(ef_mean.cpu().numpy())
            cls_true_all.extend(cls_true.numpy()); cls_prob_all.extend(torch.sigmoid(cls_logit).cpu().numpy())
    mae = mean_absolute_error(ef_true_all, ef_pred_all)
    try:
        auc = roc_auc_score(cls_true_all, cls_prob_all)
    except ValueError:
        auc = float('nan')
    return {'mae': mae, 'auc': auc}

def run_ablation(config_name, use_delta, use_film, n_epochs=6, checkpoint_every_minutes=15):
    model = EchoAblation(use_delta, use_film, pretrained_backbone_path=PRETRAINED_BACKBONE_PATH).to(device)
    with torch.no_grad():
        mean_ef = float(train_df[ef_col].mean())
        std_ef = float(train_df[ef_col].std())
        model.ef_mean_head.bias.fill_(mean_ef); model.ef_mean_head.weight.zero_()
        model.ef_q_lo_head.bias.fill_(mean_ef - std_ef); model.ef_q_lo_head.weight.zero_()
        model.ef_q_hi_head.bias.fill_(mean_ef + std_ef); model.ef_q_hi_head.weight.zero_()

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    pos_rate = (train_df[ef_col] < HFrEF_THRESHOLD).mean()
    pos_weight = torch.tensor([(1 - pos_rate) / max(pos_rate, 1e-6)], device=device)
    cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    state_path = os.path.join(PROJECT_ROOT, 'checkpoints', f'ablation_{config_name}', 'latest.pt')
    os.makedirs(os.path.dirname(state_path), exist_ok=True)
    start_epoch, resume_step = 0, 0
    if os.path.exists(state_path):
        state = torch.load(state_path, map_location=device)
        model.load_state_dict(state['model_state']); optimizer.load_state_dict(state['optimizer_state'])
        start_epoch, resume_step = state['epoch'], state['step_in_epoch']
        print(f'[{config_name}] resumed from epoch={start_epoch}')

    last_ckpt_time = time.time()
    for epoch in range(start_epoch, n_epochs):
        model.train()
        for step, (pair, ef_true, cls_true, quality) in enumerate(train_loader):
            if epoch == start_epoch and step < resume_step:
                continue
            pair, ef_true, cls_true, quality = pair.to(device), ef_true.to(device), cls_true.to(device), quality.to(device)
            ef_mean, q_lo, q_hi, cls_logit = model(pair, quality)
            loss = (F.l1_loss(ef_mean, ef_true)
                    + pinball_loss(ef_true, q_lo, TAU_LO)
                    + pinball_loss(ef_true, q_hi, TAU_HI)
                    + cls_criterion(cls_logit, cls_true))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            if (time.time() - last_ckpt_time) > checkpoint_every_minutes * 60:
                torch.save({'epoch': epoch, 'step_in_epoch': step + 1, 'model_state': model.state_dict(),
                            'optimizer_state': optimizer.state_dict()}, state_path)
                last_ckpt_time = time.time()
        resume_step = 0
        torch.save({'epoch': epoch + 1, 'step_in_epoch': 0, 'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict()}, state_path)

    val_metrics = evaluate(model, val_loader)
    print(f'[{config_name}] val_mae={val_metrics["mae"]:.3f} val_auc={val_metrics["auc"]:.3f}')
    return val_metrics

## Run all 4 configurations (VAL only -- test set not touched)

Each is resumable independently (separate checkpoint folders), so a disconnect only costs the in-progress configuration, not the others already completed.

In [ ]:
configs = [
    ('baseline_concat', False, False),
    ('plus_delta',      True,  False),
    ('plus_film',       False, True),
    ('full_model',      True,  True),
]

results = {}
for name, use_delta, use_film in configs:
    print(f'--- Running {name} (delta={use_delta}, FiLM={use_film}) ---')
    results[name] = run_ablation(name, use_delta, use_film)

results_df = pd.DataFrame(results).T
results_df.index.name = 'configuration'
print('\n', results_df)
results_df.to_csv(os.path.join(PROJECT_ROOT, 'logs', 'component_ablation_results.csv'))
print('\nSaved to logs/component_ablation_results.csv')

## What to report

This produces a clean 2x2 ablation table for the paper. Expected reportable patterns (report whichever actually occurs -- do not adjust the narrative to fit an assumption):
- If both `plus_delta` and `plus_film` individually improve over `baseline_concat`, and `full_model` is best of all, this is the cleanest possible story: each component helps, and they combine additively.
- If one component helps and the other does not (or slightly hurts) in isolation but the combination still works, that is also a valid, reportable finding -- interaction effects between architectural components are common and worth discussing honestly rather than dropping the weaker-seeming component from the narrative post hoc.
- This ablation uses a shorter training schedule (6 epochs) than the final model's full training, for speed across 4 runs; absolute numbers may differ slightly from the fully-trained final model, but the *relative ordering* between configurations is what matters for this analysis.